In [1]:
from google.colab import files
uploaded= files.upload()

Saving retail_store_sales.csv to retail_store_sales.csv


In [4]:
# Import the libraries required for data anaysis
import pandas as pd
import numpy as np

print("successfully imported")

successfully imported


In [5]:
# Load the CSV file into a Pandas DataFrame
df=pd.read_csv("retail_store_sales.csv")
print("Dataset loaded successfully.")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")

Dataset loaded successfully.
Total rows: 12575
Total columns: 11


In [6]:
# Display the first five rows to understand the structure of the dataset
df.head()

,Transaction ID,Customer ID,Category,Item,Price Per Unit,Quantity,Total Spent,Payment Method,Location,Transaction Date,Discount Applied
0,TXN_6867343,CUST_09,Patisserie,Item_10_PAT,18.5,10.0,185.0,Digital Wallet,Online,2024-04-08,True
1,TXN_3731986,CUST_22,Milk Products,Item_17_MILK,29.0,9.0,261.0,Digital Wallet,Online,2023-07-23,True
2,TXN_9303719,CUST_02,Butchers,Item_12_BUT,21.5,2.0,43.0,Credit Card,Online,2022-10-05,False
3,TXN_9458126,CUST_06,Beverages,Item_16_BEV,27.5,9.0,247.5,Credit Card,Online,2022-05-07,NaN
4,TXN_4575373,CUST_05,Food,Item_6_FOOD,12.5,7.0,87.5,Digital Wallet,Online,2022-10-02,False


In [7]:
# Display basic statistical information for numerical columns
df.describe()

,Price Per Unit,Quantity,Total Spent
count,11966.000000,11971.000000,11971.000000
mean,23.365912,5.536380,129.652577
std,10.743519,2.857883,94.750697
min,5.000000,1.000000,5.000000
25%,14.000000,3.000000,51.000000
50%,23.000000,6.000000,108.500000
75%,33.500000,8.000000,192.000000
max,41.000000,10.000000,410.000000


In [8]:
# Display column names, data types, and non-null values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12575 entries, 0 to 12574
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Transaction ID    12575 non-null  object 
 1   Customer ID       12575 non-null  object 
 2   Category          12575 non-null  object 
 3   Item              11362 non-null  object 
 4   Price Per Unit    11966 non-null  float64
 5   Quantity          11971 non-null  float64
 6   Total Spent       11971 non-null  float64
 7   Payment Method    12575 non-null  object 
 8   Location          12575 non-null  object 
 9   Transaction Date  12575 non-null  object 
 10  Discount Applied  8376 non-null   object 
dtypes: float64(3), object(8)
memory usage: 1.1+ MB


In [9]:
# Check the number of missing values in each column

missing_values = df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
Transaction ID         0
Customer ID            0
Category               0
Item                1213
Price Per Unit       609
Quantity             604
Total Spent          604
Payment Method         0
Location               0
Transaction Date       0
Discount Applied    4199
dtype: int64


In [10]:
# Check the missing-value pattern of Price, Quantity, and Total Spent

missing_pattern = df[
    ['Price Per Unit', 'Quantity', 'Total Spent']
].isnull()

print("Missing-value combinations:")
print(missing_pattern.value_counts())

Missing-value combinations:
Price Per Unit  Quantity  Total Spent
False           False     False          11362
True            False     False            609
False           True      True             604
Name: count, dtype: int64


In [11]:
# Calculate missing Price Per Unit using Total Spent and Quantity

mask = (
    df['Price Per Unit'].isnull() &
    df['Quantity'].notnull() &
    df['Total Spent'].notnull()
)

print("Rows where Price Per Unit can be calculated:", mask.sum())

df.loc[mask, 'Price Per Unit'] = (
    df.loc[mask, 'Total Spent'] / df.loc[mask, 'Quantity']
)

print("Remaining missing Price Per Unit values:", df['Price Per Unit'].isnull().sum())

Rows where Price Per Unit can be calculated: 609
Remaining missing Price Per Unit values: 0


In [12]:
# Identify rows where both Quantity and Total Spent are missing

problem_rows = df[
    df['Quantity'].isnull() &
    df['Total Spent'].isnull()
]

print("Rows with both Quantity and Total Spent missing:", len(problem_rows))

Rows with both Quantity and Total Spent missing: 604


In [13]:
# Remove rows where both Quantity and Total Spent are missing

before = len(df)

df = df.dropna(
    subset=['Quantity', 'Total Spent'],
    how='all'
)

after = len(df)

print("Non-recoverable rows removed:", before - after)
print("Remaining rows in dataset:", after)

Non-recoverable rows removed: 604
Remaining rows in dataset: 11971


In [14]:
# Verify missing values after removing non-recoverable rows

print("Remaining missing Quantity values:",
      df['Quantity'].isnull().sum())

print("Remaining missing Total Spent values:",
      df['Total Spent'].isnull().sum())

Remaining missing Quantity values: 0
Remaining missing Total Spent values: 0


In [15]:
# Check whether missing Item values can be identified using Category and Price Per Unit

missing_items = df[df['Item'].isnull()]

print("Rows with missing Item:", len(missing_items))

missing_items[['Category', 'Price Per Unit', 'Quantity', 'Total Spent']].head(10)

Rows with missing Item: 609


,Category,Price Per Unit,Quantity,Total Spent
5,Patisserie,20.0,10.0,200.0
11,Milk Products,6.5,8.0,52.0
17,Milk Products,27.5,10.0,275.0
21,Milk Products,35.0,3.0,105.0
32,Food,24.5,8.0,196.0
127,Butchers,27.5,10.0,275.0
159,Butchers,14.0,9.0,126.0
216,Patisserie,5.0,8.0,40.0
247,Electric household essentials,21.5,7.0,150.5
271,Milk Products,11.0,9.0,99.0


In [16]:
# Check which Category and Price combinations have known Item names

known_items = df.dropna(subset=['Item'])

item_mapping = (
    known_items
    .groupby(['Category', 'Price Per Unit'])['Item']
    .nunique()
)

print("Category-Price combinations with a known Item:", len(item_mapping))
print("Combinations mapped to exactly one Item:", (item_mapping == 1).sum())
print("Combinations mapped to multiple Items:", (item_mapping > 1).sum())

Category-Price combinations with a known Item: 200
Combinations mapped to exactly one Item: 200
Combinations mapped to multiple Items: 0


In [17]:
# Create a mapping from Category and Price Per Unit to Item

known_items = df.dropna(subset=['Item'])

item_mapping = (
    known_items
    .drop_duplicates(subset=['Category', 'Price Per Unit'])
    .set_index(['Category', 'Price Per Unit'])['Item']
)

print("Unique Category-Price-Item mappings created:", len(item_mapping))

Unique Category-Price-Item mappings created: 200


In [18]:
# Fill missing Item values using the Category and Price Per Unit mapping

item_values = pd.Series(
    df.set_index(['Category', 'Price Per Unit']).index.map(item_mapping),
    index=df.index
)

df['Item'] = df['Item'].fillna(item_values)

print("Remaining missing Item values:", df['Item'].isnull().sum())

Remaining missing Item values: 0


In [19]:
# Check the values present in Discount Applied

print("Values in Discount Applied:")
print(df['Discount Applied'].value_counts(dropna=False))

Values in Discount Applied:
Discount Applied
True     4019
NaN      3988
False    3964
Name: count, dtype: int64


In [20]:
# Compare sales values for rows with and without Discount information

print("Average Total Spent by Discount Applied:")
print(df.groupby('Discount Applied', dropna=False)['Total Spent'].mean())

Average Total Spent by Discount Applied:
Discount Applied
False    129.953330
True     130.491043
NaN      128.508651
Name: Total Spent, dtype: float64


In [21]:
# Compare average Price Per Unit and Quantity by Discount Applied

print("Average Price Per Unit by Discount Applied:")
print(df.groupby('Discount Applied', dropna=False)['Price Per Unit'].mean())

print("\nAverage Quantity by Discount Applied:")
print(df.groupby('Discount Applied', dropna=False)['Quantity'].mean())

Average Price Per Unit by Discount Applied:
Discount Applied
False    23.343592
True     23.464668
NaN      23.273445
Name: Price Per Unit, dtype: float64

Average Quantity by Discount Applied:
Discount Applied
False    5.580979
True     5.533466
NaN      5.494985
Name: Quantity, dtype: float64


In [22]:
# Fill missing Discount Applied values

df['Discount Applied'] = df['Discount Applied'].fillna("Unknown")

print("Remaining missing Discount Applied values:",
      df['Discount Applied'].isnull().sum())

print("\nDiscount Applied values after cleaning:")
print(df['Discount Applied'].value_counts())

Remaining missing Discount Applied values: 0

Discount Applied values after cleaning:
Discount Applied
True       4019
Unknown    3988
False      3964
Name: count, dtype: int64


In [23]:
print("Current number of rows:", len(df))

Current number of rows: 11971


In [24]:
# Check for duplicate rows

duplicate_rows = df.duplicated().sum()

print("Duplicate rows found:", duplicate_rows)

Duplicate rows found: 0


In [25]:
# Convert Transaction Date to datetime format

df['Transaction Date'] = pd.to_datetime(df['Transaction Date'])

print("Transaction Date data type:", df['Transaction Date'].dtype)

Transaction Date data type: datetime64[ns]


In [26]:
# Quality Check : Check data types of all columns

print("Data types of each column:")
print(df.dtypes)

Data types of each column:
Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                   float64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied            object
dtype: object


In [27]:
# Quality Check : Check for remaining missing values

missing_values = df.isnull().sum()

print("Remaining missing values in each column:")
print(missing_values)

Remaining missing values in each column:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
dtype: int64


In [28]:
# Quality Check : Check for duplicate rows

duplicate_rows = df.duplicated().sum()

print("Duplicate rows found:", duplicate_rows)

Duplicate rows found: 0


In [29]:
# Quality Check : Check for invalid Quantity values

invalid_quantity = df[df['Quantity'] <= 0]

print("Rows with zero or negative Quantity:", len(invalid_quantity))

Rows with zero or negative Quantity: 0


In [30]:
# Check for actual decimal Quantity values, excluding missing values

decimal_quantities = df.loc[
    df['Quantity'].notna() & (df['Quantity'] % 1 != 0),
    'Quantity'
]

print("Number of actual decimal Quantity values:", len(decimal_quantities))

Number of actual decimal Quantity values: 0


In [31]:
# Check for missing Quantity values

print("Remaining missing Quantity values:",
      df['Quantity'].isnull().sum())

Remaining missing Quantity values: 0


In [32]:
# Convert Quantity from float to integer

df['Quantity'] = df['Quantity'].astype(int)

print("Quantity data type after conversion:", df['Quantity'].dtype)

Quantity data type after conversion: int64


In [33]:
# Verify Quantity after conversion

print("Quantity data type:", df['Quantity'].dtype)
print("Remaining missing Quantity values:", df['Quantity'].isnull().sum())
print("Rows with zero or negative Quantity:",
      (df['Quantity'] <= 0).sum())

Quantity data type: int64
Remaining missing Quantity values: 0
Rows with zero or negative Quantity: 0


In [34]:
# Quality Check : Check for invalid Price Per Unit values

invalid_price = df[df['Price Per Unit'] <= 0]

print("Rows with zero or negative Price Per Unit:", len(invalid_price))

Rows with zero or negative Price Per Unit: 0


In [35]:
# Quality Check : Check for invalid Total Spent values

invalid_total = df[df['Total Spent'] <= 0]

print("Rows with zero or negative Total Spent:", len(invalid_total))

Rows with zero or negative Total Spent: 0


In [36]:
# Quality Check : Verify Total Spent calculation

calculated_total = df['Quantity'] * df['Price Per Unit']

formula_mismatch = ~np.isclose(
    calculated_total,
    df['Total Spent']
)

print("Rows where Quantity × Price Per Unit != Total Spent:",
      formula_mismatch.sum())

Rows where Quantity × Price Per Unit != Total Spent: 0


In [37]:
# Quality Check : Check final dataset size

print("Final number of rows:", df.shape[0])
print("Final number of columns:", df.shape[1])

Final number of rows: 11971
Final number of columns: 11


In [39]:
# Create Day of Week column from Transaction Date
df['Day of Week'] = df['Transaction Date'].dt.day_name()

print("Day of Week column created successfully.")
print("\nSample values:")
print(df[['Transaction Date', 'Day of Week']].head())

Day of Week column created successfully.

Sample values:
  Transaction Date Day of Week
0       2024-04-08      Monday
1       2023-07-23      Sunday
2       2022-10-05   Wednesday
3       2022-05-07    Saturday
4       2022-10-02      Sunday


In [40]:
# Create Month column from Transaction Date
df['Month'] = df['Transaction Date'].dt.month_name()

print("Month column created successfully.")
print("\nSample values:")
print(df[['Transaction Date', 'Month']].head())

Month column created successfully.

Sample values:
  Transaction Date    Month
0       2024-04-08    April
1       2023-07-23     July
2       2022-10-05  October
3       2022-05-07      May
4       2022-10-02  October


In [41]:
# Create Year column from Transaction Date
df['Year'] = df['Transaction Date'].dt.year

print("Year column created successfully.")
print("\nSample values:")
print(df[['Transaction Date', 'Year']].head())

Year column created successfully.

Sample values:
  Transaction Date  Year
0       2024-04-08  2024
1       2023-07-23  2023
2       2022-10-05  2022
3       2022-05-07  2022
4       2022-10-02  2022


In [42]:
# Final quality check before saving cleaned CSV

print("Final dataset shape:", df.shape)
print("\nColumn list:")
print(df.columns.tolist())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

Final dataset shape: (11971, 14)

Column list:
['Transaction ID', 'Customer ID', 'Category', 'Item', 'Price Per Unit', 'Quantity', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date', 'Discount Applied', 'Day of Week', 'Month', 'Year']

Missing values per column:
Transaction ID      0
Customer ID         0
Category            0
Item                0
Price Per Unit      0
Quantity            0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
Discount Applied    0
Day of Week         0
Month               0
Year                0
dtype: int64

Data types:
Transaction ID              object
Customer ID                 object
Category                    object
Item                        object
Price Per Unit             float64
Quantity                     int64
Total Spent                float64
Payment Method              object
Location                    object
Transaction Date    datetime64[ns]
Discount Applied            object
Day o

In [43]:
# Save final cleaned dataset

df.to_csv("cleaned_retail_sales.csv", index=False)

print("Cleaned dataset saved successfully as 'cleaned_retail_sales.csv'")
print("Final shape saved:", df.shape)

Cleaned dataset saved successfully as 'cleaned_retail_sales.csv'
Final shape saved: (11971, 14)
